In [1]:
import pandas as pd
from pathlib import Path

print("ClimaCare UAE AI — Data Inspection Started")
print("Pandas version:", pd.__version__)

ClimaCare UAE AI — Data Inspection Started
Pandas version: 3.0.5


In [3]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/raw/dubai-ae-292223/air_quality_historical.csv")

df = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Dataset loaded successfully
Number of rows: 1298
Number of columns: 12


In [4]:
print("Column names:")
print(df.columns.tolist())

Column names:
['date', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index', 'us_aqi', 'european_aqi']


In [5]:
df.head()

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-08-04,86.940000,52.715000,760.550000,112.455,32.490000,100.400000,0.515000,26.050000,2.480000,NaN,NaN
4,2022-08-05,86.758333,53.945833,737.458333,106.075,30.816667,48.083333,0.510833,25.458333,2.039583,142.85,82.15


In [6]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
date                     0
pm10                     3
pm2_5                    3
carbon_monoxide          3
nitrogen_dioxide         3
sulphur_dioxide          3
ozone                    3
aerosol_optical_depth    3
dust                     3
uv_index                 3
us_aqi                   4
european_aqi             4
dtype: int64


In [7]:
missing_rows = df[df.isnull().any(axis=1)]

print("Number of rows containing at least one missing value:", len(missing_rows))

missing_rows

Number of rows containing at least one missing value: 4


,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-08-04,86.94,52.715,760.55,112.455,32.49,100.4,0.515,26.05,2.48,NaN,NaN


In [11]:
from pathlib import Path
import pandas as pd

raw_folder = Path("../data/raw")

weather_files = list(raw_folder.glob("open-meteo*.csv"))

print("Weather files found:", weather_files)

Weather files found: [PosixPath('../data/raw/open-meteo-25.06N55.28E27m-2.csv')]


In [14]:
weather_df = pd.read_csv(weather_files[0], skiprows=3)

# Remove any repeated header rows
weather_df = weather_df[weather_df["time"] != "time"].copy()

print("Weather rows after removing repeated headers:", len(weather_df))
print(weather_df["time"].head())
print(weather_df["time"].tail())

Weather rows after removing repeated headers: 32450
0    2022-08-01T00:00
1    2022-08-01T01:00
2    2022-08-01T02:00
3    2022-08-01T03:00
4    2022-08-01T04:00
Name: time, dtype: str
32446    2026-02-14
32447    2026-02-15
32448    2026-02-16
32449    2026-02-17
32450    2026-02-18
Name: time, dtype: str


In [16]:
# Keep only hourly rows (rows containing "T" in the time value)
weather_hourly = weather_df[
    weather_df["time"].astype(str).str.contains("T", na=False)
].copy()

# Convert hourly timestamps to datetime
weather_hourly["time"] = pd.to_datetime(
    weather_hourly["time"],
    format="%Y-%m-%dT%H:%M"
)

# Create a daily date column
weather_hourly["date"] = weather_hourly["time"].dt.date

# Convert weather measurement columns to numeric
weather_hourly["temperature_2m (°C)"] = pd.to_numeric(
    weather_hourly["temperature_2m (°C)"], errors="coerce"
)

weather_hourly["relative_humidity_2m (%)"] = pd.to_numeric(
    weather_hourly["relative_humidity_2m (%)"], errors="coerce"
)

weather_hourly["wind_speed_10m (km/h)"] = pd.to_numeric(
    weather_hourly["wind_speed_10m (km/h)"], errors="coerce"
)

# Calculate one average value per day
weather_daily = weather_hourly.groupby("date").agg({
    "temperature_2m (°C)": "mean",
    "relative_humidity_2m (%)": "mean",
    "wind_speed_10m (km/h)": "mean"
}).reset_index()

# Rename the columns to simpler names
weather_daily = weather_daily.rename(columns={
    "temperature_2m (°C)": "temperature",
    "relative_humidity_2m (%)": "humidity",
    "wind_speed_10m (km/h)": "wind_speed"
})

print("Hourly rows used:", len(weather_hourly))
print("Daily weather rows created:", len(weather_daily))

weather_daily.head()

Hourly rows used: 31152
Daily weather rows created: 1298


,date,temperature,humidity,wind_speed
0,2022-08-01,37.304167,39.083333,15.300000
1,2022-08-02,37.300000,38.583333,10.700000
2,2022-08-03,36.762500,40.875000,10.154167
3,2022-08-04,36.183333,55.083333,10.983333
4,2022-08-05,35.033333,58.791667,10.570833


In [17]:
print("Air-quality date range:")
print(df["date"].min(), "to", df["date"].max())

print("\nWeather date range:")
print(weather_daily["date"].min(), "to", weather_daily["date"].max())

print("\nAir-quality rows:", len(df))
print("Weather daily rows:", len(weather_daily))

Air-quality date range:
2022-08-01 to 2026-02-18

Weather date range:
2022-08-01 to 2026-02-18

Air-quality rows: 1298
Weather daily rows: 1298


In [18]:
# Convert both date columns to the same datetime type
df["date"] = pd.to_datetime(df["date"])
weather_daily["date"] = pd.to_datetime(weather_daily["date"])

# Merge air-quality and weather datasets using date
climacare_df = pd.merge(
    df,
    weather_daily,
    on="date",
    how="inner"
)

print("ClimaCare merged dataset created successfully!")
print("Rows:", climacare_df.shape[0])
print("Columns:", climacare_df.shape[1])

print("\nColumns:")
print(climacare_df.columns.tolist())

climacare_df.head()

ClimaCare merged dataset created successfully!
Rows: 1298
Columns: 15

Columns:
['date', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index', 'us_aqi', 'european_aqi', 'temperature', 'humidity', 'wind_speed']


,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi,temperature,humidity,wind_speed
0,2022-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.304167,39.083333,15.300000
1,2022-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.300000,38.583333,10.700000
2,2022-08-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.762500,40.875000,10.154167
3,2022-08-04,86.940000,52.715000,760.550000,112.455,32.490000,100.400000,0.515000,26.050000,2.480000,NaN,NaN,36.183333,55.083333,10.983333
4,2022-08-05,86.758333,53.945833,737.458333,106.075,30.816667,48.083333,0.510833,25.458333,2.039583,142.85,82.15,35.033333,58.791667,10.570833


In [19]:
print("Missing values in merged ClimaCare dataset:")
print(climacare_df.isnull().sum())

print("\nTotal missing values:")
print(climacare_df.isnull().sum().sum())

Missing values in merged ClimaCare dataset:
date                     0
pm10                     3
pm2_5                    3
carbon_monoxide          3
nitrogen_dioxide         3
sulphur_dioxide          3
ozone                    3
aerosol_optical_depth    3
dust                     3
uv_index                 3
us_aqi                   4
european_aqi             4
temperature              0
humidity                 0
wind_speed               0
dtype: int64

Total missing values:
35


In [20]:
# Remove rows containing missing values
climacare_clean = climacare_df.dropna().copy()

# Reset row index
climacare_clean.reset_index(drop=True, inplace=True)

# Check the cleaned dataset
print("Original rows:", len(climacare_df))
print("Clean rows:", len(climacare_clean))
print("Rows removed:", len(climacare_df) - len(climacare_clean))

print("\nRemaining missing values:")
print(climacare_clean.isnull().sum().sum())

print("\nClean dataset shape:")
print(climacare_clean.shape)

climacare_clean.head()

Original rows: 1298
Clean rows: 1294
Rows removed: 4

Remaining missing values:
0

Clean dataset shape:
(1294, 15)


,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi,temperature,humidity,wind_speed
0,2022-08-05,86.758333,53.945833,737.458333,106.075000,30.816667,48.083333,0.510833,25.458333,2.039583,142.850000,82.150000,35.033333,58.791667,10.570833
1,2022-08-06,105.070833,66.250000,889.500000,137.662500,34.558333,43.291667,0.513333,30.125000,2.131250,152.083333,88.041667,36.204167,45.416667,10.158333
2,2022-08-07,150.691667,93.758333,813.583333,119.616667,23.904167,120.000000,0.573333,41.041667,1.889583,180.291667,98.416667,37.495833,33.208333,12.337500
3,2022-08-08,103.125000,62.745833,607.625000,94.858333,18.945833,152.666667,0.353750,29.250000,2.272917,207.166667,109.583333,38.366667,35.583333,10.900000
4,2022-08-09,76.595833,45.358333,602.625000,107.912500,24.212500,63.375000,0.339167,26.083333,2.366667,146.791667,85.625000,36.750000,44.541667,11.462500


In [21]:
from pathlib import Path

# Path for processed data
processed_folder = Path("../data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

# Save cleaned merged dataset
output_file = processed_folder / "climacare_clean.csv"

climacare_clean.to_csv(output_file, index=False)

print("Dataset saved successfully!")
print("Saved to:", output_file)
print("Final shape:", climacare_clean.shape)

Dataset saved successfully!
Saved to: ../data/processed/climacare_clean.csv
Final shape: (1294, 15)
